# Demo VIB-Net trên Kaggle (GPU)

Mô phỏng bài **"Towards Universal AI-Generated Image Detection by Variational Information Bottleneck Network"** (CVPR 2025).

**Câu hỏi:** chỉ huấn luyện trên **SD15 vs ảnh thật**, bộ phân loại có khối VIB có tổng quát hoá sang generator **chưa gặp** tốt hơn bộ phân loại tuyến tính trên CLIP (kiểu Univfd) không?

### Cách chạy trên Kaggle
1. **Settings → Accelerator: GPU** (T4 hoặc P100).
2. **Settings → Internet: On** (để clone repo tác giả, tải dataset từ HuggingFace và trọng số CLIP).
3. **Run All**.

So với bản chạy CPU (`VIBNet_demo.ipynb`), bản này:
- dùng dữ liệu lớn hơn khoảng 10 lần (2000 ảnh/lớp để huấn luyện, 500 ảnh mỗi generator lạ);
- trích đặc trưng CLIP bằng **fp16 trên GPU**;
- thêm **augmentation blur + JPEG** như bài báo, **5 bản augment** cho mỗi ảnh train;
- mục 5 huấn luyện với **batch 256, cosine LR, KL warm-up 10 epoch, early stopping** trên tập validation cùng phân phối;
- cuối mục 5 **nạp trọng số vừa huấn luyện vào class `CLIPModel` gốc của tác giả** và chạy thử 1 batch để chứng minh tương thích;
- mục 7 vẽ UMAP trên tập **Unseen** (generator chưa gặp), không dùng lại tập validation.

> Chạy ngoài Kaggle (không có `/kaggle`) thì notebook tự chuyển sang **cấu hình nhỏ** để thử nhanh trên CPU.

## 1. Cấu hình và môi trường

In [ ]:
import io, os, sys, time, json, random, subprocess, copy, types
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

IS_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUICK = not IS_KAGGLE          # đổi thành False nếu muốn chạy cấu hình đầy đủ ngoài Kaggle

CFG = dict(
    N_TRAIN_SHARDS=14,         # Tiny-GenImage có 14 shard train, mỗi shard ~143 ảnh SD15 + 1000 ảnh thật
    N_VAL_SHARDS=4,            # 4 shard validation, mỗi shard 125 ảnh/generator + 875 ảnh thật
    N_TRAIN_PER_CLASS=2000,    # ảnh thật / ảnh SD15 để huấn luyện
    N_UNSEEN_PER_GEN=500,      # ảnh giả mỗi generator lạ (ghép với số ảnh thật bằng nhau)
    N_AUG=5,                   # số bản augment blur/JPEG cho mỗi ảnh train (ngoài bản gốc)
    VAL_FRAC=0.15,             # tỉ lệ ảnh train giữ lại làm validation (early stopping)
    BATCH_EXTRACT=128,
    EPOCHS=100, PATIENCE=10, BATCH=256, LR=2e-4,
    KL_WARMUP=10,              # số epoch tăng tuyến tính hệ số KL từ 0 lên beta
    SEEDS=[0, 1, 2],
    BETAS=[0.0, 1e-4, 1e-3, 1e-2, 1e-1],
)
if QUICK:
    CFG.update(N_TRAIN_SHARDS=2, N_VAL_SHARDS=1, N_TRAIN_PER_CLASS=60,
               N_UNSEEN_PER_GEN=20, N_AUG=1, BATCH_EXTRACT=32)
NUM_WORKERS = 4 if IS_KAGGLE else 0   # notebook trên Windows không dùng được worker

print("Kaggle:", IS_KAGGLE, "| device:", DEVICE, "| QUICK:", QUICK)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps(CFG, indent=1))

In [ ]:
# Thư viện còn thiếu trên Kaggle
def ensure(pkg, mod):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, mod in [("ftfy", "ftfy"), ("regex", "regex"), ("umap-learn", "umap"),
                 ("huggingface_hub", "huggingface_hub"), ("pyarrow", "pyarrow")]:
    ensure(pkg, mod)

# Repo tác giả (chỉ dùng thư viện CLIP trong models/clip)
REPO_DIR = WORK / "VIBAIGCDetect" if IS_KAGGLE else Path.cwd().parent / "VIBAIGCDetect"
if not (REPO_DIR / "models" / "clip" / "clip.py").exists():   # chưa có, hoặc là thư mục submodule rỗng
    if REPO_DIR.exists() and not any(REPO_DIR.iterdir()):
        REPO_DIR.rmdir()
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/oceanzhf/VIBAIGCDetect", str(REPO_DIR)], check=True)

# clip.py của repo dùng `from pkg_resources import packaging`; setuptools mới có thể không còn pkg_resources
try:
    from pkg_resources import packaging  # noqa: F401
except ImportError:
    import packaging, packaging.version
    shim = types.ModuleType("pkg_resources"); shim.packaging = packaging
    sys.modules["pkg_resources"] = shim

sys.path.insert(0, str(REPO_DIR / "models"))
import clip
print("CLIP repo:", REPO_DIR)

### Minh chứng: vì sao phải tự viết pipeline huấn luyện

Ô dưới tự kiểm tra repo gốc của tác giả và ghi kết quả vào output:
1. Liệt kê các file `.py`. Repo chỉ có code **kiểm thử** (`main.py`), không có file train.
2. Tìm trong toàn bộ code các dấu hiệu của huấn luyện: `optimizer`, `.backward(`, `kl_divergence`, `KL`, `loss`. Nếu không thấy thì repo không có hàm mất mát VIB hay vòng lặp huấn luyện.
3. Gọi thử link tải **pretrained weights** (Google Drive) ghi trong README của tác giả. Mã HTTP 401/403 nghĩa là bị khoá quyền truy cập.

Khi viết báo cáo, chụp thêm màn hình trang Google Drive báo "You need access" và cây thư mục repo trên GitHub để làm căn cứ.

In [ ]:
import re, urllib.request, urllib.error

py_files = sorted(p.relative_to(REPO_DIR).as_posix() for p in REPO_DIR.rglob("*.py"))
print("Các file .py trong repo gốc:")
for f in py_files:
    print("  ", f)

patterns = {"optimizer": r"optim\.", "backward()": r"\.backward\(", "KL / kl_divergence": r"kl_div|\bKL\b|kl_loss",
            "hàm loss": r"\w*loss\w*\s*=[^=]|criterion|nn\.\w*Loss\(", "vòng lặp epoch": r"for\s+\w+\s+in\s+range\(.*epoch"}
print("\nTìm dấu hiệu code huấn luyện:")
for name, pat in patterns.items():
    hits = [f for f in py_files if re.search(pat, (REPO_DIR / f).read_text(encoding="utf-8", errors="ignore"), re.I)]
    print(f"  {name:20s}: {', '.join(hits) if hits else 'KHÔNG CÓ'}")

print("\nKiểm tra link pretrained weights trong README tác giả:")
for name, fid in [("Stable Diffusion v1.4", "1iaC6MRwM20RhbAFx-7152wL3xZmf8_oN"), ("ProGAN", "1CBC-b65aBQzWpueqA-RFFPAp1c5lzUkw")]:
    url = f"https://drive.google.com/file/d/{fid}/view?usp=drive_link"
    try:
        code_ = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=20).status
    except urllib.error.HTTPError as e:
        code_ = e.code
    except Exception as e:
        code_ = f"lỗi mạng: {e}"
    note = "BỊ KHOÁ (không có quyền truy cập)" if code_ in (401, 403) else ""
    print(f"  {name:22s} HTTP {code_} {note}\n    {url}")

## 2. Dữ liệu

Nguồn: [`TheKernel01/Tiny-GenImage`](https://huggingface.co/datasets/TheKernel01/Tiny-GenImage). Bộ này **không có SD v1.4** nên dùng **SD15** (cùng họ) để huấn luyện.

- **Train:** ảnh thật + ảnh SD15 từ các shard `train`.
- **Unseen:** ADM, BigGAN, GLIDE, Midjourney, VQDM, Wukong + ảnh thật từ các shard `validation` (không trùng ảnh train).

Mọi ảnh: cạnh ngắn → 256, crop giữa 224, nén JPEG q95 (giảm rò rỉ nhãn qua định dạng: GenImage gốc có ảnh thật là JPEG, ảnh giả là PNG). Trên Kaggle mỗi shard được xoá ngay sau khi đọc để tiết kiệm ổ đĩa. Tổng tải khoảng 8GB.

In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

HF_REPO = "TheKernel01/Tiny-GenImage"
GEN = {0: "Real", 1: "ADM", 2: "BigGAN", 3: "GLIDE", 4: "Midjourney", 6: "SD15", 7: "VQDM", 8: "Wukong"}
UNSEEN = ["ADM", "BigGAN", "GLIDE", "Midjourney", "VQDM", "Wukong"]
SHARD_DIR = Path("/tmp/shards")

def read_shard(fname, keep, cap, rng):
    # Chỉ giữ tối đa cap[g] ảnh mỗi nhóm trong shard này, để không giữ hàng nghìn ảnh thật thừa trong RAM
    kw = {"local_dir": SHARD_DIR} if IS_KAGGLE else {}
    path = hf_hub_download(HF_REPO, fname, repo_type="dataset", **kw)
    pf = pq.ParquetFile(path)
    gens = pf.read(columns=["generator"]).column("generator").to_pylist()
    rows = {}
    for i, g in enumerate(gens):
        if GEN[g] in keep:
            rows.setdefault(GEN[g], []).append(i)
    take = sorted(i for g, idx in rows.items() for i in rng.sample(idx, min(cap[g], len(idx))))
    # Chỉ đọc cột ảnh ở những row group có dòng cần lấy
    out, start = {}, 0
    for rg in range(pf.num_row_groups):
        n = pf.metadata.row_group(rg).num_rows
        local = [i - start for i in take if start <= i < start + n]
        if local:
            t = pf.read_row_group(rg, columns=["image", "generator"]).take(local)
            for im, g in zip(t.column("image").to_pylist(), t.column("generator").to_pylist()):
                out.setdefault(GEN[g], []).append(im["bytes"])
            del t
        start += n
    del pf
    pa.default_memory_pool().release_unused()   # trả bộ nhớ đệm của pyarrow cho hệ thống
    if IS_KAGGLE:
        os.remove(path)
    return out

def collect(prefix, n_shards, total, keep, cap, rng):
    pool = {}
    for i in range(n_shards):
        for g, items in read_shard(f"data/{prefix}-{i:05d}-of-{total:05d}.parquet", keep, cap, rng).items():
            pool.setdefault(g, []).extend(items)
        print(f"  {prefix} shard {i + 1}/{n_shards}:", {g: len(v) for g, v in pool.items()}, flush=True)
    return pool

def per_shard(n, shards):
    return -(-n // shards) + 5   # chia đều cho các shard, cộng dư một ít

t0 = time.time()
rng = random.Random(0)
n_trs, n_vas = CFG["N_TRAIN_SHARDS"], CFG["N_VAL_SHARDS"]
tr_cap = {g: per_shard(CFG["N_TRAIN_PER_CLASS"], n_trs) for g in ["Real", "SD15"]}
un_cap = {g: per_shard(CFG["N_UNSEEN_PER_GEN"], n_vas) for g in UNSEEN}
un_cap["Real"] = per_shard(2 * CFG["N_UNSEEN_PER_GEN"], n_vas)
tr_pool = collect("train", n_trs, 14, {"Real", "SD15"}, tr_cap, rng)
un_pool = collect("validation", n_vas, 4, {"Real", *UNSEEN}, un_cap, rng)
for v in list(tr_pool.values()) + list(un_pool.values()):
    rng.shuffle(v)

n_tr = min(CFG["N_TRAIN_PER_CLASS"], len(tr_pool["SD15"]), len(tr_pool["Real"]))
n_un = min([CFG["N_UNSEEN_PER_GEN"]] + [len(un_pool[g]) for g in UNSEEN])
n_un_real = min(2 * n_un, len(un_pool["Real"]))      # pool ảnh thật để ghép cân bằng với từng generator
train_raw = tr_pool["Real"][:n_tr] + tr_pool["SD15"][:n_tr]
y_train = np.array([0] * n_tr + [1] * n_tr)
unseen_raw = un_pool["Real"][:n_un_real] + sum((un_pool[g][:n_un] for g in UNSEEN), [])
gen_unseen = np.array(["Real"] * n_un_real + sum(([g] * n_un for g in UNSEEN), []))
del tr_pool, un_pool
gc.collect()
print(f"train: {n_tr} thật + {n_tr} SD15 | unseen: {n_un_real} thật + {n_un}/generator | {time.time() - t0:.0f}s")

## 3. Trích đặc trưng CLIP ViT-L/14 (GPU, fp16)

CLIP được **đóng băng** như trong bài, nên chỉ trích đặc trưng **một lần**. Ảnh train có thêm `N_AUG = 5` bản augment theo kiểu Wang 2020 mà bài dùng, tức mỗi ảnh có 6 phiên bản (1 gốc + 5 augment), để MLP bớt overfit:
- 50% làm mờ Gaussian, σ ∈ [0, 3]
- 50% nén JPEG, chất lượng ∈ [30, 100]

Mỗi bản augment được trích đặc trưng riêng, nên lúc huấn luyện vẫn chỉ là các phép tính trên vector 768 chiều, rất nhanh. Kết quả được lưu cache vào `features_kaggle.npz`. Cache cũ (ví dụ trích với `N_AUG = 2`) bị **xoá và trích lại** tự động khi cấu hình thay đổi.

In [ ]:
import torchvision.transforms as T

MEAN = [0.48145466, 0.4578275, 0.40821073]
STD = [0.26862954, 0.26130258, 0.27577711]
TO_TENSOR = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

def normalize_img(raw):
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    w, h = img.size; s = 256 / min(w, h)
    img = img.resize((max(224, round(w * s)), max(224, round(h * s))), Image.BICUBIC)
    w, h = img.size; l, t = (w - 224) // 2, (h - 224) // 2
    img = img.crop((l, t, l + 224, t + 224))
    buf = io.BytesIO(); img.save(buf, "JPEG", quality=95)
    return Image.open(buf).convert("RGB")

def augment(img, r):
    if r.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=r.uniform(0, 3)))
    if r.random() < 0.5:
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=r.randint(30, 100))
        img = Image.open(buf).convert("RGB")
    return img

class ImgDS(torch.utils.data.Dataset):
    def __init__(self, raws, aug_seed=None):
        self.raws, self.aug_seed = raws, aug_seed
    def __len__(self):
        return len(self.raws)
    def __getitem__(self, i):
        img = normalize_img(self.raws[i])
        if self.aug_seed is not None:
            img = augment(img, random.Random(self.aug_seed * 1_000_003 + i))
        return TO_TENSOR(img)

clip_model, _ = clip.load("ViT-L/14", device=DEVICE)   # trên GPU, CLIP tự chuyển sang fp16
clip_model.eval()
_text_forward = clip_model.transformer.forward             # repo trả về (out, x) -> encode_text hỏng
clip_model.transformer.forward = lambda x: _text_forward(x)[1]

@torch.no_grad()
def extract(raws, aug_seed=None):
    dl = torch.utils.data.DataLoader(ImgDS(raws, aug_seed), batch_size=CFG["BATCH_EXTRACT"],
                                     num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")
    feats = []
    for x in dl:
        feats.append(clip_model.encode_image(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(feats).numpy()

CACHE = WORK / "features_kaggle.npz"
sig = json.dumps({k: CFG[k] for k in ["N_TRAIN_SHARDS", "N_VAL_SHARDS", "N_TRAIN_PER_CLASS", "N_UNSEEN_PER_GEN", "N_AUG"]})
if CACHE.exists() and str(np.load(CACHE)["sig"]) != sig:
    CACHE.unlink()   # cache cũ trích với cấu hình khác (vd N_AUG=2) -> xoá, trích lại
    print("đã xoá cache cũ khác cấu hình")
if CACHE.exists():
    d = np.load(CACHE); X_views, X_un = d["X_views"], d["X_un"]
    print("dùng cache", CACHE)
else:
    t0 = time.time()
    X_views = np.stack([extract(train_raw)] + [extract(train_raw, aug_seed=v) for v in range(1, CFG["N_AUG"] + 1)])
    print(f"train: {X_views.shape} trong {time.time() - t0:.0f}s", flush=True)
    t0 = time.time()
    X_un = extract(unseen_raw)
    print(f"unseen: {X_un.shape} trong {time.time() - t0:.0f}s")
    np.savez(CACHE, X_views=X_views, X_un=X_un, sig=sig)

with torch.no_grad():
    txt = clip_model.encode_text(clip.tokenize(["a photo of an animal", "a photo of a man-made object"]).to(DEVICE)).float()
    TXT = (txt / txt.norm(dim=-1, keepdim=True)).cpu().numpy()

## 4. Kiến trúc VIB-head và hàm mất mát

Information Bottleneck: $\mathcal{L}_{IB} = I(Z;Y) - \beta\, I(Z;X)$. Xấp xỉ biến phân (công thức 6 của bài) cho hàm cần **cực tiểu hoá**:

$$\mathcal{L}_{VIB} = \frac{1}{N}\sum_{n}\mathbb{E}_{z\sim p_\theta(z|x_n)}\big[-\log q(y_n|z)\big] + \beta\,\mathrm{KL}\big[p_\theta(z|x_n)\,\|\,\mathcal{N}(0,I)\big]$$

`VIBHead` lặp lại đúng `CLIPModel` trong `models/clip_models.py` của repo: dropout 0.5 → `fc_1` 768→1024 → `fc_2` 1024→1024 → `fc_3` 1024→512 → μ, σ = softplus(· − 5) với K = 256 → `decode` 256→1. Phần KL và β do ta viết theo bài, vì repo chỉ có code kiểm thử. Khi đánh giá dùng z = μ để kết quả ổn định.

**Vì sao vẫn có `VIBHead` riêng?** `CLIPModel` của tác giả gói cả CLIP lẫn phần đầu phân loại, nên mỗi bước huấn luyện phải chạy lại CLIP (rất chậm). `VIBHead` chính là phần đầu phân loại của `CLIPModel` (cùng tên tầng, cùng kích thước), tách ra để huấn luyện trên đặc trưng đã cache. Ô dưới **import trực tiếp `CLIPModel` từ `models/clip_models.py` của tác giả**, và cuối mục 5 sẽ nạp trọng số đã huấn luyện ngược vào class gốc này.

In [ ]:
K = 256

class VIBHead(nn.Module):
    def __init__(self, in_dim=768, k=K):
        super().__init__()
        self.k = k
        self.dropout = nn.Dropout(0.5)
        self.fc_1 = nn.Linear(in_dim, 1024)
        self.fc_2 = nn.Linear(1024, 1024)
        self.fc_3 = nn.Linear(1024, 2 * k)
        self.decode = nn.Linear(k, 1)

    def encode(self, x):
        h = F.relu(self.fc_1(self.dropout(x)))
        h = F.relu(self.fc_2(h))
        s = self.fc_3(h)
        return s[:, :self.k], F.softplus(s[:, self.k:] - 5)

    def forward(self, x, sample=True):
        mu, std = self.encode(x)
        z = mu + torch.randn_like(std) * std if sample else mu
        return mu, std, self.decode(z).squeeze(1)

class LinearHead(nn.Module):
    # Baseline kiểu Univfd: một lớp tuyến tính trên đặc trưng CLIP
    def __init__(self, in_dim=768):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, x, sample=True):
        return None, None, self.fc(x).squeeze(1)

def kl_to_std_normal(mu, std):
    return (0.5 * (mu ** 2 + std ** 2 - 1 - 2 * torch.log(std + 1e-8))).sum(1).mean()

In [ ]:
# Import trực tiếp class CLIPModel của tác giả từ VIBAIGCDetect/models/clip_models.py.
# Nạp file như một module con của gói giả "author_models" để KHÔNG chạy models/__init__.py
# (file đó import thêm các backbone ImageNet không cần thiết, dễ lỗi với torchvision mới).
import importlib.util

_pkg = types.ModuleType("author_models"); _pkg.__path__ = [str(REPO_DIR / "models")]
sys.modules["author_models"] = _pkg
_spec = importlib.util.spec_from_file_location("author_models.clip_models", REPO_DIR / "models" / "clip_models.py")
author_clip_models = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = author_clip_models
_spec.loader.exec_module(author_clip_models)
CLIPModel = author_clip_models.CLIPModel
print("Đã import:", CLIPModel, "từ", _spec.origin)

# Tham số phần đầu phân loại của VIBHead; cuối mục 5 sẽ đối chiếu với CLIPModel gốc
print("Tham số VIBHead:", {k: tuple(v.shape) for k, v in VIBHead().state_dict().items()})

## 5. Huấn luyện (tối ưu cho GPU)

Những điểm khác bản CPU:

| | Bản CPU | Bản Kaggle |
|---|---|---|
| Dữ liệu train | 160 ảnh/lớp, không augment | ~1700 ảnh/lớp × (1 + `N_AUG`) bản augment |
| Batch | 64 | **256** (như bài) |
| Lịch học | lr cố định 2e-4, 60 epoch | lr 2e-4 + **cosine**, tối đa 100 epoch |
| Hệ số KL | β cố định từ epoch đầu | **KL warm-up**: β tăng tuyến tính từ 0 lên β trong 10 epoch đầu |
| Dừng | hết epoch | **early stopping** theo BCE trên validation (chỉ xét sau khi hết warm-up), khôi phục trọng số tốt nhất |
| Dữ liệu trên GPU | không | đặt toàn bộ tensor lên GPU một lần, không dùng DataLoader |

Validation là 15% ảnh train (**cùng phân phối SD15**), chỉ dùng bản gốc. Các bản augment của cùng một ảnh luôn nằm cùng một phía của phép chia, để không rò rỉ. Tập unseen **không** được dùng để chọn epoch hay β.

**KL warm-up:** nếu phạt KL đủ mạnh ngay từ epoch đầu, phân phối của z bị ép về N(0, I) trước khi mô hình kịp học đặc trưng phân biệt thật/giả. Vì vậy hệ số được tăng dần: `curr_beta = beta * min(1.0, (ep + 1) / 10)`. Early stopping chỉ bắt đầu xét sau warm-up, để không chọn nhầm trọng số lúc β còn nhỏ.

Cuối mục này nạp trọng số `vibnet` vào **class `CLIPModel` gốc của tác giả** và chạy thử một batch.

In [ ]:
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

def split_train(seed):
    idx = np.arange(len(y_train))
    return train_test_split(idx, test_size=CFG["VAL_FRAC"], stratify=y_train, random_state=seed)

def make_fit(fit_idx, use_aug=True):
    views = X_views if use_aug else X_views[:1]
    Xf = views[:, fit_idx].reshape(-1, X_views.shape[-1])
    yf = np.tile(y_train[fit_idx], len(views))
    return Xf, yf

def to_dev(a, dtype=torch.float32):
    return torch.as_tensor(a, dtype=dtype, device=DEVICE)

def train_head(model, Xf, yf, Xv, yv, beta=0.0, seed=0, history=None):
    torch.manual_seed(seed)
    model = model.to(DEVICE)
    Xf, yf, Xv, yv = to_dev(Xf), to_dev(yf), to_dev(Xv), to_dev(yv)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS"])
    best, best_state, bad, n = float("inf"), None, 0, len(Xf)
    for ep in range(CFG["EPOCHS"]):
        curr_beta = beta * min(1.0, (ep + 1) / CFG["KL_WARMUP"])   # KL warm-up tuyến tính
        model.train()
        perm, tot = torch.randperm(n, device=DEVICE), 0.0
        for i in range(0, n, CFG["BATCH"]):
            idx = perm[i:i + CFG["BATCH"]]
            mu, std, logit = model(Xf[idx])
            loss = F.binary_cross_entropy_with_logits(logit, yf[idx])
            if mu is not None and beta > 0:
                loss = loss + curr_beta * kl_to_std_normal(mu, std)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item() * len(idx)
        sched.step()
        model.eval()
        with torch.no_grad():
            vloss = F.binary_cross_entropy_with_logits(model(Xv, sample=False)[2], yv).item()
        if history is not None:
            history.append((tot / n, vloss))
        if ep + 1 < CFG["KL_WARMUP"]:
            continue   # đang warm-up: chưa xét early stopping
        if vloss < best - 1e-4:
            best, best_state, bad = vloss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]:
                break
    model.load_state_dict(best_state)
    return model, ep + 1

@torch.no_grad()
def predict(model, X):
    model.eval()
    return torch.sigmoid(model(to_dev(X), sample=False)[2]).cpu().numpy()

def ap_acc(y, p, thr=0.5):
    return average_precision_score(y, p), float(((p > thr) == (y == 1)).mean())

In [ ]:
# Huấn luyện thử một lần: Linear vs VIB (beta = 0.01), seed 0
fit_idx, val_idx = split_train(0)
Xf, yf = make_fit(fit_idx)
Xv, yv = X_views[0, val_idx], y_train[val_idx]
print(f"train: {len(Xf)} vector ({len(fit_idx)} ảnh × {X_views.shape[0]} bản) | val: {len(Xv)} ảnh")

hist_lin, hist_vib = [], []
t0 = time.time()
lin, ep_lin = train_head(LinearHead(), Xf, yf, Xv, yv, beta=0.0, seed=0, history=hist_lin)
vibnet, ep_vib = train_head(VIBHead(), Xf, yf, Xv, yv, beta=1e-2, seed=0, history=hist_vib)
print(f"xong trong {time.time() - t0:.1f}s | Linear dừng ở epoch {ep_lin}, VIB dừng ở epoch {ep_vib}")
for name, net in [("Linear", lin), ("VIB beta=0.01", vibnet)]:
    ap, acc = ap_acc(yv, predict(net, Xv))
    print(f"{name:14s} validation SD15: AP={ap * 100:.1f}  ACC={acc * 100:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (name, h) in zip(axes, [("Linear (BCE)", hist_lin), ("VIB (BCE + β·KL)", hist_vib)]):
    h = np.array(h)
    ax.plot(h[:, 0], label="train loss"); ax.plot(h[:, 1], label="val BCE")
    ax.axvline(int(np.argmin(h[:, 1])), color="gray", ls="--", lw=0.8, label="epoch tốt nhất")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.legend()
axes[0].set_ylabel("loss")
plt.tight_layout(); plt.show()

### Kiểm tra: nạp trọng số vào `CLIPModel` gốc của tác giả

1. Khởi tạo `CLIPModel("ViT-L/14")` đúng class của repo (tự nạp CLIP qua `models/clip`).
2. Đối chiếu tên và kích thước tham số của phần đầu phân loại với `VIBHead`.
3. Chép trọng số `fc_1`, `fc_2`, `fc_3`, `decode` từ `vibnet` vừa huấn luyện bằng `load_state_dict`.
4. Chạy 1 batch 16 ảnh unseen qua `forward` **gốc** của tác giả, so với `vibnet` trên cùng đặc trưng CLIP.

Nếu μ, σ và logit trùng nhau (sai số cỡ 1e-5, do làm tròn số thực) thì mô hình tự huấn luyện **tương thích hoàn toàn** với code của bài báo: có thể lưu và dùng lại bằng chính `main.py` của tác giả.

In [ ]:
author = CLIPModel("ViT-L/14")                         # class gốc: CLIP + fc_1/fc_2/fc_3/decode
author_head = {k: tuple(v.shape) for k, v in author.state_dict().items() if not k.startswith("model.")}
ours_head = {k: tuple(v.shape) for k, v in vibnet.state_dict().items()}
print("Tham số phần đầu của CLIPModel gốc:", author_head)
assert author_head == ours_head, "Kiến trúc không khớp!"
print("=> Tên và kích thước tham số khớp 100% với VIBHead")

for name in ["fc_1", "fc_2", "fc_3", "decode"]:
    getattr(author, name).load_state_dict(getattr(vibnet, name).state_dict())
author = author.to(DEVICE).eval()
vibnet.eval()

idx = np.random.RandomState(0).choice(len(unseen_raw), 16, replace=False)
x = torch.stack([TO_TENSOR(normalize_img(unseen_raw[i])) for i in idx]).to(DEVICE)
with torch.no_grad():
    (mu_a, std_a), logit_a = author(x)                 # forward GỐC của tác giả (z được lấy mẫu)
    feat = author.model.encode_image(x).float()        # đặc trưng CLIP mà CLIPModel dùng bên trong
    mu_o, std_o, logit_o = vibnet(feat, sample=False)  # head tự huấn luyện, z = mu
    logit_a_mu = author.decode(mu_a).squeeze(1)        # head của tác giả, z = mu
    p16 = predict(vibnet, clip_model.encode_image(x).float().cpu().numpy())  # pipeline mục 3 (fp16)

print(f"max |mu_tác giả - mu_ta|          = {(mu_a - mu_o).abs().max().item():.2e}")
print(f"max |std_tác giả - std_ta|        = {(std_a - std_o).abs().max().item():.2e}")
print(f"max |logit_tác giả - logit_ta|    = {(logit_a_mu - logit_o).abs().max().item():.2e}   (cùng z = mu)")
print(f"max |logit lấy mẫu - logit z=mu|  = {(logit_a.squeeze(1) - logit_a_mu).abs().max().item():.2e}   (do nhiễu z = mu + eps*std)")
pa_ = torch.sigmoid(logit_a_mu).cpu().numpy()
print(f"max |P(giả) tác giả - P(giả) pipeline fp16 mục 3| = {np.abs(pa_ - p16).max():.3f}")
print("Dự đoán thật/giả trùng nhau:", int(((pa_ > 0.5) == (p16 > 0.5)).sum()), "/", len(idx))
assert torch.allclose(mu_a, mu_o, atol=1e-4) and torch.allclose(logit_a_mu, logit_o, atol=1e-4)
print("=> Head tự huấn luyện chạy đúng trong CLIPModel gốc của tác giả")

torch.save({"model": author.state_dict()}, WORK / "vibnet_clipmodel.pth")   # cùng định dạng main.py đọc
print("Đã lưu checkpoint dạng tác giả:", WORK / "vibnet_clipmodel.pth")
del author, feat, x
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 6. Thí nghiệm chính: tổng quát hoá sang generator chưa gặp

- Linear và VIB với β ∈ {0, 1e-4, 1e-3, 1e-2, 1e-1}, tất cả có augmentation.
- Thêm hai dòng **không augmentation** để xem tác dụng của blur/JPEG mà bài nhấn mạnh.
- 3 seed, báo mean ± std. Mỗi generator được đánh giá trên tập **cân bằng 1:1**, nên **mức ngẫu nhiên = 50%**.

In [ ]:
models = {"Linear (Univfd-like)": (LinearHead, 0.0, True)}
models.update({f"VIB beta={b:g}": (VIBHead, b, True) for b in CFG["BETAS"]})
models["Linear, không augment"] = (LinearHead, 0.0, False)
models["VIB beta=0.01, không augment"] = (VIBHead, 1e-2, False)

real_un = np.where(gen_unseen == "Real")[0]
fake_un = {g: np.where(gen_unseen == g)[0] for g in UNSEEN}
rows, t0 = [], time.time()
for seed in CFG["SEEDS"]:
    rng_np = np.random.RandomState(seed)
    fit_idx, val_idx = split_train(seed)
    Xv, yv = X_views[0, val_idx], y_train[val_idx]
    eval_sets = {}
    for g in UNSEEN:   # cùng một tập đánh giá cho mọi mô hình trong seed này
        idx = np.concatenate([rng_np.choice(real_un, len(fake_un[g]), replace=False), fake_un[g]])
        eval_sets[g] = (X_un[idx], (gen_unseen[idx] != "Real").astype(int))
    for name, (cls, beta, use_aug) in models.items():
        Xf, yf = make_fit(fit_idx, use_aug)
        net, ep = train_head(cls(), Xf, yf, Xv, yv, beta=beta, seed=seed)
        ap, acc = ap_acc(yv, predict(net, Xv))
        rows.append(dict(model=name, seed=seed, test="In-dist (SD15)", AP=ap * 100, ACC=acc * 100, epochs=ep))
        for g, (Xe, ye) in eval_sets.items():
            ap, acc = ap_acc(ye, predict(net, Xe))
            rows.append(dict(model=name, seed=seed, test=g, AP=ap * 100, ACC=acc * 100, epochs=ep))
    print(f"seed {seed} xong ({time.time() - t0:.0f}s)", flush=True)

res = pd.DataFrame(rows)
um = res[res.test.isin(UNSEEN)].groupby(["model", "seed"])[["AP", "ACC"]].mean().reset_index()
um["test"] = "Unseen mean"
res = pd.concat([res, um], ignore_index=True)
res.to_csv(WORK / "results_kaggle.csv", index=False)
order = ["In-dist (SD15)"] + UNSEEN + ["Unseen mean"]

def table(metric):
    g = res.groupby(["model", "test"])[metric]
    t = (g.mean().round(1).astype(str) + " ± " + g.std(ddof=0).round(1).astype(str)).unstack()[order]
    return t.loc[list(models)]

print("AP (%)"); display(table("AP"))
print("ACC (%) @ ngưỡng 0.5"); display(table("ACC"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8.5), sharex=True)
w = 0.85 / len(models)
for ax, metric in zip(axes, ["AP", "ACC"]):
    g = res.groupby(["model", "test"])[metric]
    mean, std = g.mean().unstack()[order], g.std(ddof=0).unstack()[order]
    for i, m in enumerate(models):
        ax.bar(np.arange(len(order)) + i * w, mean.loc[m], w, yerr=std.loc[m], capsize=2, label=m,
               hatch="//" if "không augment" in m else None)
    ax.axhline(50, color="gray", lw=0.8, ls="--")
    ax.set_ylim(30, 102); ax.set_ylabel(f"{metric} (%)")
axes[0].legend(ncol=4, fontsize=8)
axes[0].set_title("Huấn luyện SD15 vs thật → kiểm thử trên generator chưa gặp (gạch ngang = ngẫu nhiên)")
axes[1].set_xticks(np.arange(len(order)) + 0.425 - w / 2); axes[1].set_xticklabels(order, rotation=20)
plt.tight_layout(); plt.savefig(WORK / "results_kaggle.png", dpi=130); plt.show()

**Cách đọc:**
- *Linear* so với *VIB β=0*: tác dụng của MLP. *VIB β=0* so với *β>0*: tác dụng riêng của **nút thắt KL**.
- Dòng có augment so với dòng *không augment*: tác dụng của blur/JPEG.
- Chênh lệch nhỏ hơn thanh sai số thì chưa đủ để kết luận.

**Vì sao AP có thể cao mà ACC lại thấp?** ACC được tính tại **ngưỡng cứng 0.5**, đúng như bài báo. Nhưng do **lệch miền phân phối (domain shift)** giữa SD15 (dữ liệu huấn luyện) và các họ mô hình khác (như GAN), dải xác suất đầu ra trên generator lạ bị **dịch chuyển** so với mốc 0.5. Ví dụ toàn bộ ảnh BigGAN có thể nhận P(giả) quanh 0.2–0.4: mô hình vẫn xếp ảnh giả cao hơn ảnh thật (AP cao), nhưng gần như mọi ảnh đều rơi dưới 0.5 nên bị đoán là thật (ACC thấp). Vì vậy ACC rất nhạy với ngưỡng cắt; cần nhìn **AP (không phụ thuộc ngưỡng)** để đánh giá đúng khả năng phân tách đặc trưng của VIB-Net.

## 7. UMAP kiểu Hình 3 của bài báo

Nhãn "loại ảnh" là nhãn **xấp xỉ** bằng zero-shot CLIP (*animal* vs *man-made object*, chia theo trung vị).

Dữ liệu vẽ là **500 ảnh thật + 500 ảnh giả lấy ngẫu nhiên từ tập Unseen** (ADM, BigGAN, GLIDE, Midjourney, VQDM, Wukong). Tập này không dùng để huấn luyện, cũng không dùng để chọn mô hình (early stopping chỉ dùng validation SD15), nên hình khách quan hơn và có nhiều loại ảnh giả thay vì chỉ SD15.

In [ ]:
import umap
from sklearn.metrics import silhouette_score

r7 = np.random.RandomState(0)
real_i, fake_i = np.where(gen_unseen == "Real")[0], np.where(gen_unseen != "Real")[0]
n7 = min(500, len(real_i), len(fake_i))
sub = np.concatenate([r7.choice(real_i, n7, replace=False), r7.choice(fake_i, n7, replace=False)])
Fraw, ysub = X_un[sub], (gen_unseen[sub] != "Real").astype(int)
print(f"UMAP trên Unseen: {n7} thật + {n7} giả", dict(zip(*np.unique(gen_unseen[sub], return_counts=True))))
s = (Fraw / np.linalg.norm(Fraw, axis=1, keepdims=True)) @ TXT.T
diff = s[:, 1] - s[:, 0]
cat = (diff > np.median(diff)).astype(int)
vibnet.eval()
with torch.no_grad():
    Fmu = vibnet.encode(to_dev(Fraw))[0].cpu().numpy()

group = cat * 2 + ysub
names = ["animal real", "animal fake", "object real", "object fake"]
colors = ["#4c78a8", "#aec7e8", "#e45756", "#ffb3b0"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (title, Fm) in zip(axes, [("CLIP ViT-L/14 raw", Fraw), ("VIB latent mu (β=0.01)", Fmu)]):
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Fm)
    for k in range(4):
        m = group == k
        ax.scatter(emb[m, 0], emb[m, 1], s=10, c=colors[k], label=names[k])
    ax.set_title(f"{title}\nsilhouette theo loại={silhouette_score(Fm, cat):.3f}, "
                 f"theo thật/giả={silhouette_score(Fm, ysub):.3f}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Thử dự đoán trên từng ảnh

Ảnh → CLIP → head → xác suất **giả**, dùng hai head ở mục 5. Muốn thử ảnh của bạn: tải ảnh lên Kaggle (Add Input) rồi gọi `p_fake_file("/kaggle/input/.../anh.jpg")`.

In [ ]:
@torch.no_grad()
def p_fake_img(img):
    f = clip_model.encode_image(TO_TENSOR(img).unsqueeze(0).to(DEVICE)).float().cpu().numpy()
    return {"Linear": float(predict(lin, f)[0]), "VIB": float(predict(vibnet, f)[0])}

def p_fake_file(path):
    with open(path, "rb") as fh:
        img = normalize_img(fh.read())
    return img, p_fake_img(img)

r = np.random.RandomState(1)
picks = [r.choice(np.where(gen_unseen == g)[0]) for g in ["Real", "Real", "BigGAN", "Midjourney", "GLIDE", "Wukong"]]
fig, axes = plt.subplots(1, len(picks), figsize=(3 * len(picks), 3.6))
for ax, i in zip(axes, picks):
    img = normalize_img(unseen_raw[i])
    probs = p_fake_img(img)
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"nhãn: {gen_unseen[i]}\n" + "\n".join(f"{k}: P(giả)={v:.2f}" for k, v in probs.items()), fontsize=9)
plt.tight_layout(); plt.show()

## 9. Kết luận và giới hạn

Đọc lại bảng ở mục 6 để trả lời:
1. Augmentation blur/JPEG (5 bản/ảnh) có cải thiện cột *Unseen mean* không?
2. VIB β>0 có hơn β=0 và Linear vượt quá thanh sai số không?
3. Generator nào vẫn gần mức ngẫu nhiên? Bản CPU trước đây thấy ADM, VQDM, BigGAN gần 50%.

**Về chỉ số ACC và AP:** ACC giữ đúng chuẩn bài báo (ngưỡng 0.5). Tuy nhiên do **domain shift** giữa SD15 và các họ mô hình khác (như GAN), dải xác suất đầu ra trên generator lạ bị dịch khỏi mốc 0.5, nên một generator có thể có AP khá nhưng ACC thấp. Khi kết luận về khả năng phân tách đặc trưng của VIB-Net cần dựa chủ yếu vào **AP**, còn ACC phản ánh thêm việc ngưỡng 0.5 có còn phù hợp trên miền mới hay không.

**Tương thích với code gốc:** cuối mục 5 đã nạp trọng số tự huấn luyện vào `CLIPModel` của tác giả và cho kết quả trùng khớp; checkpoint `vibnet_clipmodel.pth` lưu cùng định dạng mà `main.py` của tác giả đọc (`state_dict['model']`).

**Giới hạn:**
- Vẫn nhỏ hơn bài (bài dùng 32.000 ảnh train), và dùng SD15 thay SD v1.4. Wukong cùng họ SD nên chỉ BigGAN là GAN khác họ thật sự.
- Không dùng checkpoint của tác giả. β, lịch học, số epoch không theo đúng bài vì bài không công bố đủ chi tiết.
- Nhãn "loại ảnh" trong UMAP là nhãn xấp xỉ.
- Repo tác giả không có code huấn luyện và link pretrained weights bị khoá (xem minh chứng ở mục 1), nên pipeline huấn luyện được viết lại từ công thức (6) của bài.
- Kết quả, `features_kaggle.npz`, `results_kaggle.csv`, `results_kaggle.png`, `vibnet_clipmodel.pth` được lưu ở `/kaggle/working` (tải về ở tab Output).